# Non-linearity pipeline (with HP tuning)

One notebook, two parts, run top to bottom:

**Part 1 — XGBoost HP tuning.** Seeded random search per stock × target on an early
tuning block (day-pair validation, early stopping on the validation day, scored by
mean MSE ratio vs. the zero-return benchmark). Winners — including the median
early-stopped tree count — are frozen to
`model_outputs/XGBoost/Tuning/best_params_seed<SEED>.json`, plus flat-top /
edge-of-range diagnostics. Part 2 (and `xgboost.ipynb`) consume the json; the
training pipeline starts strictly after the tuning block.

**Part 2 — Nonlinearity probes.** Quantifies how much **non-linearity** contributes
to return predictability, per stock × day × target, as a first-class model family
under `model_outputs/Nonlinearity`. Two probes per walk-forward split
(train day *i* → test day *i+1*):

1. **Hybrid** — OLS fit on the train day, plus an XGBoost booster trained on the OLS
   *train-day in-sample residuals* (frozen tuned params from Part 1,
   **including `n_estimators` — no early stopping, no test-day contact**; this is a
   deliberate honesty change vs the exploratory version in
   `ad_hoc/XGB_overfitting_exercise.ipynb`, which early-stopped on the test day).
2. **Linearity ladder** — ridge regressions on cumulative feature expansions, each rung
   adding one kind of non-linearity: `linear` → `asym` (sign asymmetry) → `curv`
   (marginal curvature) → `inter` (all pairwise interactions). The ridge penalty is
   scaled proportionally to the rung's column count (constant per-coefficient
   penalty, see D015).

All legs share one `daily_diagnostics.parquet` distinguished by a `model_leg` column
(`hybrid`, `ladder_linear`, `ladder_asym`, `ladder_curv`, `ladder_inter`).
Tick residuals are stored for the **hybrid leg only** (float32: 100ms returns sit near
float16's subnormal floor), so `evaluation_utils.load_tick_residuals` works unchanged.

OLS / XGB baselines are **not refit here**: the manifest records the run ids of the
Regression and XGBoost runs this run is meant to be compared against. The comparison
itself — ladder monotonicity, regime scatter, day-clustered significance, DM tests —
lives in **section 4 of `scripts/model_evaluation.ipynb`**.

Step 2 of the thesis (deeper order-book levels) = regenerate `data/processed` with more
`F_` columns, bump `FEATURE_SET_ID`, re-run.

In [ ]:
import os, warnings, sys
import json
import time
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_processing as du
import utils.model_utils as mu

importlib.reload(du)
importlib.reload(mu)

## Part 1 — XGBoost HP tuning (random search, freeze winners)

In [ ]:
# ============================================================
# Tuning configuration
#
# The search runs once per stock x price-measure x horizon 
# Each trial trains on day d and is scored on the FULL day d+1, averaged over the day pairs
# All winning params (incl. tree count) are frozen
# The training pipeline starts strictly after TUNE_DATES
# Cost = N_TRIALS x N_PAIRS x n_targets fits per stock
# ============================================================

TUNE_SYMBOLS = du.SYMBOLS[:1] # Stocks to tune

TUNE_DATES = du.SAMPLE_DATES[:10] # Early time block used for tuning. Excluded from headline results

N_TRIALS = 40 # Random-search trials per stock-target

N_PAIRS = 5 # (train day, validation day) pairs per trial, spread evenly over TUNE_DATES

SEED = 0

HORIZONS = ["100ms", "2s", "30s", "5m"] # Return horizons defining the targets.

# Searched hyperparameters: name -> (distribution, low, high)
# "log"/"logint" sample log-uniformly (right choice for scale-like params)
# "int"/"logint" round to integers.
SEARCH_SPACE = {
    "max_depth": ("int", 3, 8),
    "learning_rate": ("log", 0.01, 0.3),
    "min_child_weight": ("logint", 5, 100),
    "subsample": ("uniform", 0.5, 1.0),
    "colsample_bytree": ("uniform", 0.5, 1.0),
    "reg_lambda": ("log", 0.01, 10.0),
}

# Fixed (non-searched) model settings for every trial fit
# n_estimators is only an upper bound: early stopping on the validation day picks the actual tree count
BASE_PARAMS = dict(
    n_estimators=2000,
    early_stopping_rounds=50,
    eval_metric="rmse",
    tree_method="hist",
    max_bin=128,
    device=mu.select_device(),   # least-used GPU, else "cpu"
    n_jobs=-1,
    random_state=0,
)
print(f"XGBoost device: {BASE_PARAMS['device']}")

In [ ]:
# ============================================================
# XGBoost-specific search functions
# ============================================================

def sample_params(rng: np.random.Generator) -> dict:
    """Draw one random-search trial from SEARCH_SPACE."""
    params = {}
    for name, (kind, lo, hi) in SEARCH_SPACE.items():
        if kind == "int":
            params[name] = int(rng.integers(lo, hi + 1))
        elif kind == "uniform":
            params[name] = float(rng.uniform(lo, hi))
        elif kind == "log":
            params[name] = float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
        elif kind == "logint":
            params[name] = int(round(np.exp(rng.uniform(np.log(lo), np.log(hi)))))
        else:
            raise ValueError(f"Unknown distribution kind: {kind}")
    return params


def random_search(
        pairs: list,
        base_params: dict,
        n_trials: int,
        seed: int = 0,
) -> pd.DataFrame:
    """
    Seeded random search for one target (1D y) with day-pair validation.

    pairs: list of (X_tr, y_tr, X_val, y_val) tuples: train on day d,
    validate on the FULL following day d+1. This mirrors the walk-forward
    deployment (same training-set size, same one-day-ahead task) and avoids
    the pre-close bias a same-day tail slice would have (intraday volatility
    is U-shaped, so end-of-day rows are not representative).

    Each trial fits one model per pair with early stopping on the validation
    day (base_params should set a large n_estimators and
    early_stopping_rounds). The per-pair score is the MSE ratio against the
    zero-return benchmark, mse_bench / mse_model = mean(y_val^2) / best_score^2
    (best_score is the early-stopped validation RMSE; the ratio is
    scale-invariant, so scaled target units are fine). The trial's score is
    the mean ratio across pairs (higher = better), and `n_estimators_frozen`
    is the median early-stopped best_iteration: the tree count to freeze
    alongside the winning params.
    """
    rng = np.random.default_rng(seed)
    rows = []

    for trial in range(n_trials):
        params = sample_params(rng)
        ratios, iterations = [], []

        start = time.perf_counter()
        for X_tr, y_tr, X_val, y_val in pairs:
            model = XGBRegressor(**{**base_params, **params})
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            mse_bench = float(np.mean(y_val ** 2))
            ratios.append(mse_bench / float(model.best_score) ** 2)
            iterations.append(int(model.best_iteration))

        rows.append({
            "trial": trial,
            **params,
            "mean_mse_ratio": float(np.mean(ratios)),
            "n_estimators_frozen": int(np.median(iterations)),
            "fit_seconds": time.perf_counter() - start,
        })

    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# Tuning derived setup
# ============================================================
warnings.filterwarnings("ignore")

PARENT = os.path.dirname(os.getcwd())
DATA_ROOT = f"{PARENT}/data/processed"
TUNING_DIR = f"{PARENT}/model_outputs/XGBoost/Tuning"

sample_cols = pd.read_parquet(
    f"{DATA_ROOT}/{TUNE_SYMBOLS[0]}/{TUNE_DATES[0]}.parquet"
).columns
FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = [c for c in sample_cols
               if c.startswith("T_") and c.rsplit("_", 1)[-1] in HORIZONS]

# Evenly spread pair start indices over the block (pair i = days i, i+1)
PAIR_IDX = sorted(set(
    np.linspace(0, len(TUNE_DATES) - 2, N_PAIRS).round().astype(int)
))
print("day pairs:", [(TUNE_DATES[i], TUNE_DATES[i+1]) for i in PAIR_IDX])

In [ ]:
# ============================================================
# Per-stock, per-target random search (checkpointed per stock):
# each stock's trials are written to TRIALS_DIR/<symbol>.parquet as
# soon as the stock finishes, and stocks with an existing file are skipped
# ============================================================
TRIALS_DIR = f"{TUNING_DIR}/trials_seed{SEED}"

start = time.perf_counter()

for symbol in tqdm(TUNE_SYMBOLS, desc="Stocks", unit="stock"):
    if os.path.exists(f"{TRIALS_DIR}/{symbol}.parquet"):
        tqdm.write(f"{symbol}: checkpoint found, skipping")
        continue

    # Each stock is tuned on its own early block (no pooling).
    cache = mu.load_day_cache(DATA_ROOT, symbol, TUNE_DATES, FEATURE_COLS, TARGET_COLS)
    # this symbol may be missing days: spread the pairs over its available days
    tune_days = sorted(cache)
    pair_idx = sorted(set(np.linspace(0, len(tune_days) - 2, N_PAIRS).round().astype(int)))
    symbol_trials = []

    for j, target in enumerate(tqdm(TARGET_COLS, desc=symbol, leave=False, unit="target")):
        # Targets in scaled units (mu.TARGET_SCALE): raw log returns stall tree
        # growth. The MSE-ratio score is scale-invariant, so unaffected.
        pairs = [
            (cache[tune_days[i]]["X"], mu.scale_target(cache[tune_days[i]]["Y"][:, j]),
             cache[tune_days[i+1]]["X"], mu.scale_target(cache[tune_days[i+1]]["Y"][:, j]))
            for i in pair_idx
        ]

        trials = random_search(
            pairs,
            base_params=BASE_PARAMS,
            n_trials=N_TRIALS,
            seed=SEED,
        )
        trials.insert(0, "target", target)
        trials.insert(0, "symbol", symbol)
        symbol_trials.append(trials)

    del cache

    mu.save_table(
        df=pd.concat(symbol_trials, ignore_index=True),
        root_dir=TRIALS_DIR,
        filename=f"{symbol}.parquet",
    )

tqdm.write(f"TOTAL SEARCH TIME: {time.perf_counter()-start:.2f}s")

# Collect all checkpoints (incl. earlier runs') for the freeze step.
all_trials = pd.concat(
    [pd.read_parquet(f"{TRIALS_DIR}/{symbol}.parquet") for symbol in TUNE_SYMBOLS],
    ignore_index=True,
)

In [ ]:
# ============================================================
# Freeze the winners -> best_params json: {"tune_dates": [...], "params": {symbol: {target: params}}}
# tune_dates records the tuning block so the training pipeline can start strictly after it.
# ============================================================
PARAM_NAMES = list(SEARCH_SPACE)

best_rows = all_trials.loc[
    all_trials.groupby(["symbol", "target"])["mean_mse_ratio"].idxmax()
]

best_params = {}
for _, row in best_rows.iterrows():
    params = {
        name: (int(row[name]) if SEARCH_SPACE[name][0] in ("int", "logint") else float(row[name]))
        for name in PARAM_NAMES
    }
    params["n_estimators"] = max(1, int(row["n_estimators_frozen"]))
    best_params.setdefault(row["symbol"], {})[row["target"]] = params

os.makedirs(TUNING_DIR, exist_ok=True)
with open(f"{TUNING_DIR}/best_params_seed{SEED}.json", "w") as f:
    json.dump({"tune_dates": list(TUNE_DATES), "params": best_params}, f, indent=2)

best_rows[["symbol", "target", "mean_mse_ratio", "n_estimators_frozen"] + PARAM_NAMES]

In [ ]:
# ============================================================
# Sanity check on the frozen winners, aggregated per horizon
# (winners are per symbol x target; the horizon is the level at which
# systematic patterns are expected, symbols act as replications).
# Also flags params whose winners pile up at a search-space bound,
# which suggests the range should be widened.
# ============================================================
winners = best_rows.copy()
winners["horizon"] = winners["target"].str.rsplit("_", n=1).str[-1]
winners["horizon"] = pd.Categorical(winners["horizon"], categories=HORIZONS, ordered=True)

summary = (
    winners.groupby("horizon", observed=True)[PARAM_NAMES + ["n_estimators_frozen"]]
    .agg(["median", "min", "max"])
    .round(3)
)
display(summary)

# Fraction of winners within 5% (of the range, log-range for log params) of a bound
EDGE_FRAC = 0.05
for name, (kind, lo, hi) in SEARCH_SPACE.items():
    vals = winners[name].astype(float)
    if kind in ("log", "logint"):
        vals, lo, hi = np.log(vals), np.log(lo), np.log(hi)
    tol = EDGE_FRAC * (hi - lo)
    at_lo, at_hi = (vals <= lo + tol).mean(), (vals >= hi - tol).mean()
    if at_lo or at_hi:
        print(f"{name}: {at_lo:.0%} of winners at lower bound, {at_hi:.0%} at upper bound")

# n_estimators hitting the 2000 cap would mean early stopping never triggered
capped = (winners["n_estimators_frozen"] >= BASE_PARAMS["n_estimators"]).mean()
if capped:
    print(f"n_estimators_frozen at the {BASE_PARAMS['n_estimators']}-tree cap: {capped:.0%}")

In [ ]:
# ============================================================
# Within-target checks
# 1) Dispersion of the winning params across symbols for each target:
#    small spread -> a pooled/shared config per target would do;
#    large spread -> per-stock tuning earns its cost.
#    Spread is normalized by the searched range (log-range for log
#    params), so 0 = identical winners, 1 = winners span the full range.
# 2) Flatness of the search landscape per symbol x target: number of
#    trials near the best trial. Nearness is measured on the GAIN over
#    the zero-return benchmark (mean_mse_ratio - 1), not on the ratio
#    itself: ratios cluster near 1, so a band relative to the ratio
#    would swallow the whole spread. A trial ties the winner if its
#    gain is within (1 - TIE_FRAC) x |best gain| of the best -- i.e.
#    best gain x 0.9 if positive, x 1.1 if negative, so the winner
#    always passes. Many near-ties mean the exact winning values are
#    weakly identified (a flat optimum), so cross-symbol dispersion
#    in (1) should not be over-interpreted.
# ============================================================
def normalized_spread(g):
    out = {}
    for name, (kind, lo, hi) in SEARCH_SPACE.items():
        vals = g[name].astype(float)
        if kind in ("log", "logint"):
            vals, lo, hi = np.log(vals), np.log(lo), np.log(hi)
        out[name] = (vals.max() - vals.min()) / (hi - lo)
    return pd.Series(out)

spread = winners.groupby("target").apply(normalized_spread, include_groups=False).round(2)
spread["n_symbols"] = winners.groupby("target").size()
display(spread)

TIE_FRAC = 0.9  # trials capturing >= 90% of the best gain's magnitude count as near-ties

def count_near_ties(g):
    gain = g["mean_mse_ratio"] - 1.0
    best = gain.max()
    threshold = best - (1 - TIE_FRAC) * abs(best)
    return int((gain >= threshold).sum())

near_ties = (
    all_trials.groupby(["symbol", "target"])
    .apply(count_near_ties, include_groups=False)
    .rename("n_trials_near_best")
    .reset_index()
)
display(near_ties.pivot(index="symbol", columns="target", values="n_trials_near_best"))

## Part 2 — Nonlinearity probes (hybrid + linearity ladder)

In [ ]:
# ============================================================
# Nonlinearity configuration
# (HORIZONS, SEED, PARENT, DATA_ROOT, TUNING_DIR come from Part 1)
# ============================================================

RUN_SYMBOLS = du.SYMBOLS[:1]

SAVE_TICK_LEVEL_DATA = True   # hybrid-leg residuals (needed for Diebold-Mariano)

# Run manifest metadata (model_outputs/Nonlinearity/runs/<id>/manifest.json).
MODEL_FAMILY = "Nonlinearity"
FEATURE_SET_ID = "microP_l1Imb_9Lags"
PURPOSE = ""  # free text: why this run exists

# Baseline runs this run is meant to be compared against (evaluation only;
# nothing is refit from them). Recorded in the manifest for auditability.
REF_REGRESSION_RUN_ID = "00"
REF_XGB_RUN_ID = None         # set once an XGBoost run with tick residuals exists

TUNING_SEED = SEED            # which tuning json to load (written by Part 1)

# Ladder settings
CLIP_SD = 5.0                 # winsorization of standardized features (see utils)
# Ladder ridge penalty: alpha = RIDGE_ALPHA * n_cols / n_features, holding the
# per-coefficient penalty constant as the rungs widen (D015 fix 1).
RIDGE_ALPHA = 1.0             # anchor alpha_0 at the linear rung

# Fixed (non-tuned) XGB settings; merged with the tuned params per fit.
XGB_PARAMS = dict(
    tree_method="hist",
    max_bin=128,
    device=BASE_PARAMS["device"],   # same device pick as Part 1
    n_jobs=-1,
    random_state=0,
)

In [ ]:
# ============================================================
# Nonlinearity derived setup
# ============================================================
OUTPUT_ROOT = f"{PARENT}/model_outputs/Nonlinearity"

# Frozen tuned params (incl. n_estimators): {"tune_dates": [...], "params": {symbol: {target: params}}}
# Read back from the json Part 1 wrote (not from memory), so Part 2 can also run
# standalone against an earlier tuning run.
BEST_PARAMS_PATH = f"{TUNING_DIR}/best_params_seed{TUNING_SEED}.json"
if not os.path.exists(BEST_PARAMS_PATH):
    raise FileNotFoundError(f"{BEST_PARAMS_PATH} not found - run Part 1 above first.")
with open(BEST_PARAMS_PATH) as f:
    _tuning = json.load(f)
TUNED_PARAMS = _tuning["params"]
TUNE_DATES = _tuning["tune_dates"]

# Walk-forward window: strictly after the tuning block (same rule as xgboost.ipynb).
# NOTE: empty on the local data subset (the tuning block covers all local days);
# for a local test run, override SD manually, e.g. SD = du.SAMPLE_DATES[-4:]
SD = [d for d in du.SAMPLE_DATES if d > max(TUNE_DATES)]

sample_cols = pd.read_parquet(f"{DATA_ROOT}/{RUN_SYMBOLS[0]}/{SD[0]}.parquet").columns
FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = [c for c in sample_cols if c.startswith("T_") and c.rsplit("_", 1)[-1] in HORIZONS]

MODEL_LEGS = ["hybrid", "ladder_linear", "ladder_asym", "ladder_curv", "ladder_inter"]

RUN_ID, DIRS = mu.start_run(
    output_root=OUTPUT_ROOT,
    model_family=MODEL_FAMILY,
    feature_set_id=FEATURE_SET_ID,
    symbols=RUN_SYMBOLS,
    train_start=SD[0],
    test_end=SD[-1],
    horizons=HORIZONS,
    feature_cols=FEATURE_COLS,
    target_cols=TARGET_COLS,
    training_scheme="walk-forward: train day i, test day i+1",
    purpose=PURPOSE,
    hp_config={
        "ref_regression_run_id": REF_REGRESSION_RUN_ID,
        "ref_xgboost_run_id": REF_XGB_RUN_ID,
        "tuning": {
            "source": BEST_PARAMS_PATH,
            "seed": TUNING_SEED,
            "tune_dates": TUNE_DATES,
            "fixed_params": {k: v for k, v in XGB_PARAMS.items() if k != "device"},
            "params": {s: TUNED_PARAMS[s] for s in RUN_SYMBOLS},
        },
        "ladder": {"clip_sd": CLIP_SD, "ridge_alpha": RIDGE_ALPHA,
                   "alpha_rule": "alpha = ridge_alpha * n_cols / n_features"},
        "model_legs": MODEL_LEGS,
        "tick_residual_legs": ["hybrid"],
        "tick_residual_dtype": "float32",
    },
    save_tick_level_data=SAVE_TICK_LEVEL_DATA,
)

RUN_DIR = DIRS["run"]
TICK_OUTPUT_DIR = DIRS["tick"]

daily_results = []

In [ ]:
# ============================================================
# Main Loop
# ============================================================
GLOBAL_START = time.perf_counter()

for symbol in tqdm(RUN_SYMBOLS, desc="Processing symbols", unit="symbol"):
    day_cache = mu.load_day_cache(DATA_ROOT, symbol, SD, FEATURE_COLS, TARGET_COLS)
    sd_sym = sorted(day_cache)  # days actually available for this symbol
    symbol_params = TUNED_PARAMS[symbol]   # {target: frozen params incl. n_estimators}

    for i in tqdm(range(len(sd_sym) - 1), desc=f"{symbol}", leave=False, unit="split"):
        train_day, test_day = sd_sym[i], sd_sym[i + 1]
        train_cache, test_cache = day_cache[train_day], day_cache[test_day]
        X_train, X_test = train_cache["X"], test_cache["X"]
        Y_train, Y_test = train_cache["Y"], test_cache["Y"]

        # ---- OLS stage (refit inline; cheap). Fit in scaled target units
        # (mu.TARGET_SCALE, a no-op for OLS) so the train residuals feeding the
        # booster are already scaled — raw log returns stall XGBoost tree growth.
        ols = LinearRegression().fit(X_train, mu.scale_target(Y_train))
        ols_resid_train_scaled = (mu.scale_target(Y_train) - ols.predict(X_train)).astype(np.float32)
        ols_pred_test = mu.unscale_prediction(ols.predict(X_test)).astype(np.float32)

        # ---- Ladder inputs (target-independent, shared across targets)
        Z_train, Z_test, _, _ = mu.standardize_features(X_train, X_test, clip_sd=CLIP_SD)
        blocks = mu.ladder_blocks(Z_train, Z_test)

        # ---- Per-leg residual matrices (raw log-return units)
        leg_resid = {leg: np.empty_like(Y_test) for leg in MODEL_LEGS}

        for j, target in enumerate(TARGET_COLS):
            # Hybrid: booster on OLS train residuals, frozen params, no early stopping
            booster = XGBRegressor(**{**XGB_PARAMS, **symbol_params[target]})
            booster.fit(X_train, ols_resid_train_scaled[:, j])
            hybrid_pred = ols_pred_test[:, j] + mu.unscale_prediction(booster.predict(X_test))
            leg_resid["hybrid"][:, j] = Y_test[:, j] - hybrid_pred

            # Ladder: ridge per cumulative rung, penalty scaled with column count
            for rung, (A_train, A_test) in blocks.items():
                alpha = RIDGE_ALPHA * A_train.shape[1] / Z_train.shape[1]
                model = Ridge(alpha=alpha).fit(A_train, mu.scale_target(Y_train[:, j]))
                leg_resid[f"ladder_{rung}"][:, j] = Y_test[:, j] - mu.unscale_prediction(model.predict(A_test))

        # ---- Persist: diagnostics for every leg, tick residuals for hybrid only
        if SAVE_TICK_LEVEL_DATA:
            mu.save_tick_residuals(
                resid=leg_resid["hybrid"],
                timestamps=test_cache["timestamp"],
                target_cols=TARGET_COLS,
                tick_output_dir=TICK_OUTPUT_DIR,
                test_day=test_day,
                symbol=symbol,
                dtype=np.float32,
            )

        for leg in MODEL_LEGS:
            rows = mu.daily_diagnostic_rows(
                resid=leg_resid[leg],
                Y_test=Y_test,
                target_cols=TARGET_COLS,
                train_day=train_day,
                test_day=test_day,
                symbol=symbol,
                run_id=RUN_ID,
                n_train=X_train.shape[0],
                n_test=X_test.shape[0],
            )
            daily_results += [dict(r, model_leg=leg) for r in rows]

tqdm.write(f"\nTOTAL PIPELINE TIME: {time.perf_counter()-GLOBAL_START:.2f}s")

# ============================================================
# Save Outputs
# ============================================================
mu.save_table(
    df=pd.DataFrame(daily_results),
    root_dir=RUN_DIR,
    filename="daily_diagnostics.parquet",
)
mu.finalize_run(RUN_DIR)

In [ ]:
daily_out = pd.read_parquet(f"{RUN_DIR}/daily_diagnostics.parquet")
display(daily_out.groupby("model_leg")["mse_ratio"].describe().round(4))